In [23]:
import warnings
from typing import List

from transformers import pipeline
from sentence_transformers import SentenceTransformer

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Enforce CPU usage (-1 for Transformers pipeline, 'cpu' for SentenceTransformers)
DEVICE_ID = -1
DEVICE_STR = "cpu"

print("Loading models... (Evaluating on CPU)")

# Load Task 1 Model: Zero-shot classification
# Using facebook/bart-large-mnli as requested
classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=DEVICE_ID
)

# Load Task 2 Model: Semantic embeddings
# Using all-MiniLM-L6-v2 for generating 384-dimensional vectors
embedder = SentenceTransformer(
    "all-MiniLM-L6-v2", 
    device=DEVICE_STR
)

# Pre-define candidate labels for classification
CANDIDATE_LABELS = [
    "Roads & Infrastructure", 
    "Sanitation & Waste", 
    "Water & Plumbing", 
    "Electrical & Lighting", 
    "Public Safety"
]

print("Models loaded successfully.")

Loading models... (Evaluating on CPU)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7046.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded successfully.


In [24]:
import warnings
from typing import List

from transformers import pipeline
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

DEVICE_ID = -1
DEVICE_STR = "cpu"

print("Loading models... (Evaluating on CPU)")

classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=DEVICE_ID
)

embedder = SentenceTransformer(
    "all-MiniLM-L6-v2", 
    device=DEVICE_STR
)

# 1. Label Descriptive Tuning (Giving the model more semantic 'surface area' to match against)
CANDIDATE_LABELS = [
    "Roads, Potholes & Infrastructure", 
    "Sanitation, Garbage & Waste Management", 
    "Water Supply, Leakage & Plumbing", 
    "Electricity, Street Lights & Electrical Wiring", 
    "Public Safety, Security & Unauthorized Activity"
]

print("Models loaded successfully.")

def process_text_branch(text: str) -> dict:
    """
    Executes two sequential NLP tasks:
    1. Zero-shot classification for category prediction using a custom NLI hypothesis.
    2. Vectorization mapping the text to semantic embeddings.
    """
    if not text or not text.strip():
        raise ValueError("Input text cannot be empty or whitespace.")
        
    clean_text = text.strip()
    
    # 2. The Hypothesis Template
    # We force the model to evaluate WHO should fix it, not just WHAT words are in the text.
   # Change your classification call to this:
    classification_out = classifier(
        clean_text, 
        CANDIDATE_LABELS, 
        multi_label=False,
        hypothesis_template="This municipal complaint is regarding {}."
    )
    
    top_label = classification_out["labels"][0]
    top_score = classification_out["scores"][0]
    
    # 3. Confidence Thresholding
    # The Penalty: If the model is guessing (low confidence), we override it.
    if top_score < 0.45:
        top_label = "Ambiguous / Manual Routing Required"
    
    # Task 2: Semantic Embeddings
    embedding_array = embedder.encode(clean_text)
    embedding_list = embedding_array.tolist()
    
    return {
        "category": top_label,
        "confidence": float(top_score),
        "embedding": embedding_list
    }


Loading models... (Evaluating on CPU)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5397.06it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded successfully.


In [27]:
if __name__ == "__main__":
    
    test_input = "the path was literally dirty."
    
    try:
        result = process_text_branch(test_input)
        
        print(f"Complaint: '{test_input}'")
        print(f"Routed Category: {result['category']}")
        print(f"Confidence: {result['confidence']:.2f}")
        print(f"Embedding length: {len(result['embedding'])}")
        
    except Exception as e:
        print(f"Error: {e}")

Complaint: 'the path was literally dirty.'
Routed Category: Ambiguous / Manual Routing Required
Confidence: 0.33
Embedding length: 384
